# Allen-Cahn score-only recovery - Kaggle T4x2

This notebook consumes the outputs from rodrimandub/af-pikans-allen-cahn-t4x2-20260927 through Kaggle kernel_sources. It scores the ten existing final checkpoints against the saved pools and the already-generated GPU reference candidate. It does not train, regenerate the reference, tune, or choose checkpoints.

The notebook searches under /kaggle/input for the source campaign tree and, if necessary, its allen-cahn-artifacts.zip. It stops with an explicit missing-artifact report unless all ten checkpoints, all five saved pools, the candidate reference, and its status file are present and consistent. Results are written under /kaggle/working/allen-cahn-score-only/.

Internet is disabled and no packages are installed. The first code cell arms a 2,100-second hard watchdog, leaving 300 seconds within the 2,400-second Kaggle quota cap for setup and closeout. Scoring uses CUDA on the two visible T4s, alternately assigning endpoints to cuda:0 and cuda:1.

The reference remains candidate-only: reference_accepted=false and scientific_accepted=false are written into the score outputs. This is confirmatory scoring of held-out endpoints and must not be used to tune the models.

In [ ]:
import time
SESSION_STARTED = time.perf_counter()
import os
import signal
import threading
import sys
import json
import csv
import hashlib
import base64
import shutil
import zipfile
import importlib.util
from pathlib import Path, PurePosixPath

WATCHDOG_SECONDS = 2100.0
SESSION_QUOTA_CAP_SECONDS = 2400.0
SESSION_DEADLINE = SESSION_STARTED + WATCHDOG_SECONDS
def _hard_stop():
    os.kill(os.getpid(), signal.SIGTERM)
watchdog = threading.Timer(WATCHDOG_SECONDS, _hard_stop)
watchdog.daemon = True
watchdog.start()

RUNNER_PAYLOAD = "..."
SESSION_DIR = Path("/kaggle/working/allen-cahn-score-only") / ("run-" + str(time.time_ns()))
RESULTS_DIR = SESSION_DIR / "scores"
SESSION_DIR.mkdir(parents=True, exist_ok=False)
RESULTS_DIR.mkdir(parents=True, exist_ok=False)

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def check_deadline(where):
    if time.perf_counter() >= SESSION_DEADLINE:
        raise TimeoutError("2100-second score-only watchdog reached during " + where)

def safe_extract_relevant(archive_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    relevant = []
    with zipfile.ZipFile(archive_path, "r") as archive:
        for info in archive.infolist():
            member = PurePosixPath(info.filename)
            parts = member.parts
            if not parts or info.is_dir():
                continue
            name = parts[-1]
            keep = (
                name == "ac_gpu_reference_candidate.npz"
                or (name == "reference_status.json" and "reference" in parts)
                or name == "checkpoint-final.pt"
                or (name.startswith("seed-") and name.endswith(".npz") and "pools" in parts)
                or name == "campaign_results.json"
            )
            if not keep:
                continue
            if member.is_absolute() or ".." in parts:
                raise RuntimeError("Unsafe path in source output archive: " + info.filename)
            target = (destination.joinpath(*parts)).resolve()
            root = destination.resolve()
            if os.path.commonpath((str(root), str(target))) != str(root):
                raise RuntimeError("Unsafe path in source output archive: " + info.filename)
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info, "r") as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)
            relevant.append(str(target))
    return relevant

def campaign_roots(search_roots):
    found = set()
    for search_root in search_roots:
        if not search_root.exists():
            continue
        for candidate in search_root.rglob("ac_gpu_reference_candidate.npz"):
            if candidate.parent.name == "reference":
                found.add(candidate.parent.parent.resolve())
    return sorted(found)

SEEDS = (401, 402, 403, 404, 405)
ARMS = ("AC-RBF-Sobol", "AC-DeepXDE-MLP")

def missing_for(root):
    missing = []
    if not (root / "reference" / "ac_gpu_reference_candidate.npz").is_file():
        missing.append("reference/ac_gpu_reference_candidate.npz")
    if not (root / "reference" / "reference_status.json").is_file():
        missing.append("reference/reference_status.json")
    for seed in SEEDS:
        if not (root / "pools" / f"seed-{seed}.npz").is_file():
            missing.append(f"pools/seed-{seed}.npz")
        for arm in ARMS:
            if not (root / arm / f"seed-{seed}" / "checkpoint-final.pt").is_file():
                missing.append(f"{arm}/seed-{seed}/checkpoint-final.pt")
    return missing

def fail_discovery(message, details=None):
    report = {
        "status": "blocked_missing_source_artifact",
        "source_kernel": "rodrimandub/af-pikans-allen-cahn-t4x2-20260927",
        "message": message,
        "details": details,
        "training_repeated": False,
        "reference_accepted": False,
        "scientific_accepted": False
    }
    write_json(SESSION_DIR / "artifact_discovery_failure.json", report)
    raise FileNotFoundError(message)

try:
    import numpy as np
    import scipy
    import torch

    if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
        raise RuntimeError("Expected exactly two visible CUDA devices for Kaggle T4x2; CPU fallback is disabled.")
    gpu_names = [torch.cuda.get_device_name(index) for index in range(2)]
    if any("T4" not in name.upper() for name in gpu_names):
        raise RuntimeError("Expected two Tesla T4 devices, found " + repr(gpu_names))
    torch.set_num_threads(2)
    torch.set_default_dtype(torch.float64)
    torch.backends.cuda.matmul.allow_tf32 = False

    runner_path = SESSION_DIR / "run_campaign_source.py"
    runner_bytes = base64.b64decode(RUNNER_PAYLOAD)
    runner_path.write_bytes(runner_bytes)
    runner_sha256 = sha256_file(runner_path)
    module_spec = importlib.util.spec_from_file_location("ac_campaign_scoring_source", runner_path)
    campaign = importlib.util.module_from_spec(module_spec)
    module_spec.loader.exec_module(campaign)

    input_root = Path("/kaggle/input")
    if not input_root.is_dir():
        fail_discovery("Kaggle source input is absent at /kaggle/input; attach the completed kernel output.")
    search_roots = [input_root]
    roots = campaign_roots(search_roots)
    complete = [root for root in roots if not missing_for(root)]
    extracted_files = []
    if not complete:
        archive_dir = SESSION_DIR / "source-archive-extracts"
        for archive_path in sorted(input_root.rglob("*.zip")):
            check_deadline("source artifact discovery")
            try:
                with zipfile.ZipFile(archive_path, "r") as archive:
                    has_candidate = any(
                        PurePosixPath(name).name == "ac_gpu_reference_candidate.npz"
                        and "reference" in PurePosixPath(name).parts
                        for name in archive.namelist()
                    )
                if not has_candidate:
                    continue
                destination = archive_dir / (archive_path.stem + "-" + sha256_file(archive_path)[:12])
                extracted_files.extend(safe_extract_relevant(archive_path, destination))
                search_roots.append(destination)
            except zipfile.BadZipFile:
                continue
        roots = campaign_roots(search_roots)
        complete = [root for root in roots if not missing_for(root)]

    if not complete:
        details = {
            "input_entries": [str(path) for path in list(input_root.iterdir())[:40]],
            "candidate_campaign_roots": [
                {"root": str(root), "missing": missing_for(root)} for root in roots
            ],
            "source_archives_extracted": extracted_files[:80],
            "expected_checkpoint_pattern": "<campaign>/<arm>/seed-<401..405>/checkpoint-final.pt",
            "expected_reference": "<campaign>/reference/ac_gpu_reference_candidate.npz",
            "expected_pools": "<campaign>/pools/seed-<401..405>.npz"
        }
        fail_discovery(
            "Could not find one complete Allen-Cahn source campaign under /kaggle/input. "
            "Expected the attached source kernel outputs with all ten checkpoints, five pools, and reference.",
            details
        )
    if len(complete) != 1:
        fail_discovery(
            "Found multiple complete source campaigns; refusing to choose one implicitly.",
            {"campaign_roots": [str(root) for root in complete]}
        )
    campaign_root = complete[0]
    check_deadline("source artifact loading")

    reference_path = campaign_root / "reference" / "ac_gpu_reference_candidate.npz"
    reference_status_path = campaign_root / "reference" / "reference_status.json"
    reference_status = json.loads(reference_status_path.read_text(encoding="utf-8"))
    reference_sha256 = sha256_file(reference_path)
    if reference_status.get("field_score_status") != "candidate_only":
        raise RuntimeError("The source reference is not marked candidate_only; scoring is blocked.")
    if reference_status.get("reference_ready_for_review") is not True:
        raise RuntimeError("The source reference did not pass its numerical gates; scoring is blocked.")
    if reference_status.get("reference_accepted") is not False:
        raise RuntimeError("Refusing to upgrade or reuse an accepted reference in this score-only package.")
    if reference_status.get("candidate_sha256") != reference_sha256:
        raise RuntimeError("The candidate-reference SHA-256 does not match reference_status.json.")
    with np.load(reference_path, allow_pickle=False) as archive:
        reference = {name: archive[name].copy() for name in ("x", "t", "u")}
    if reference["u"].shape != (len(reference["t"]), len(reference["x"])):
        raise RuntimeError("Reference arrays have inconsistent dimensions.")

    source_results_path = campaign_root / "campaign_results.json"
    source_status_by_endpoint = {}
    if source_results_path.is_file():
        source_results = json.loads(source_results_path.read_text(encoding="utf-8"))
        for row in source_results.get("scores", []):
            if isinstance(row, dict) and "arm" in row and "seed" in row:
                source_status_by_endpoint[f"{row['arm']}|{row['seed']}"] = row.get("status")

    expected = []
    pool_hashes = {}
    for seed in SEEDS:
        pool_path = campaign_root / "pools" / f"seed-{seed}.npz"
        with np.load(pool_path, allow_pickle=False) as archive:
            pools = {name: archive[name].copy() for name in ("interior", "heldout")}
        if pools["interior"].shape != (8192, 2) or pools["heldout"].shape != (16384, 2):
            raise RuntimeError(f"Unexpected saved pool dimensions for seed {seed}.")
        pool_hashes[str(seed)] = sha256_file(pool_path)
        for arm in ARMS:
            checkpoint_path = campaign_root / arm / f"seed-{seed}" / "checkpoint-final.pt"
            expected.append({
                "seed": seed,
                "arm": arm,
                "checkpoint_path": checkpoint_path,
                "checkpoint_sha256": sha256_file(checkpoint_path),
                "pool_path": pool_path,
                "pools": pools
            })

    run_manifest = {
        "status": "score_only_running",
        "source_kernel": "rodrimandub/af-pikans-allen-cahn-t4x2-20260927",
        "source_campaign_root": str(campaign_root),
        "source_training_status_by_endpoint": source_status_by_endpoint,
        "source_runner_sha256_embedded": runner_sha256,
        "reference_path": str(reference_path),
        "reference_sha256": reference_sha256,
        "reference_field_score_status": "candidate_only",
        "reference_ready_for_review": True,
        "reference_accepted": False,
        "scientific_accepted": False,
        "training_repeated": False,
        "checkpoint_selection": "existing final checkpoints only; no reference-based selection",
        "confirmatory_scoring_only": True,
        "heldout_pool_hashes": pool_hashes,
        "gpu_names": gpu_names,
        "dtype": "float64",
        "scoring_device_policy": "sequential scoring, alternating cuda:0 and cuda:1",
        "watchdog_seconds_from_first_code_cell": WATCHDOG_SECONDS,
        "session_quota_cap_seconds_including_setup": SESSION_QUOTA_CAP_SECONDS,
        "internet_enabled": False
    }
    write_json(SESSION_DIR / "score_only_manifest.json", run_manifest)

    score_rows = []
    for endpoint_index, item in enumerate(expected):
        check_deadline(f"scoring {item['arm']} seed {item['seed']}")
        score_dir = RESULTS_DIR / item["arm"] / f"seed-{item['seed']}"
        score_dir.mkdir(parents=True, exist_ok=False)
        scoring_checkpoint = score_dir / "checkpoint-final.pt"
        shutil.copyfile(item["checkpoint_path"], scoring_checkpoint)
        device = torch.device(f"cuda:{endpoint_index % 2}")
        try:
            result = campaign.score_run(
                item["arm"], item["seed"], score_dir, reference, item["pools"], device, SESSION_DEADLINE
            )
            torch.cuda.synchronize(device)
            result.update({
                "post_training_score_status": "score_only_complete",
                "source_training_status": source_status_by_endpoint.get(
                    f"{item['arm']}|{item['seed']}", "not recorded in source summary"
                ),
                "source_checkpoint_sha256": item["checkpoint_sha256"],
                "saved_pool_sha256": pool_hashes[str(item["seed"])],
                "reference_candidate_sha256": reference_sha256,
                "reference_status": "candidate_only; independent review still required",
                "reference_accepted": False,
                "scientific_accepted": False,
                "training_repeated": False,
                "confirmatory_scoring_only": True,
                "heldout_test_scored": True
            })
            campaign.write_json(score_dir / "score.json", result)
            score_rows.append(result)
        except Exception as exc:
            failure = {
                "arm": item["arm"], "seed": item["seed"],
                "post_training_score_status": "score_failed_preserved",
                "exception": f"{type(exc).__name__}: {exc}",
                "source_checkpoint_sha256": item["checkpoint_sha256"],
                "reference_candidate_sha256": reference_sha256,
                "reference_accepted": False,
                "scientific_accepted": False,
                "training_repeated": False
            }
            write_json(score_dir / "score_failure.json", failure)
            score_rows.append(failure)
        write_json(SESSION_DIR / "score_only_progress.json", {
            **run_manifest, "results_completed": len(score_rows), "results": score_rows
        })

    check_deadline("score result export")
    good_rows = [row for row in score_rows if row.get("post_training_score_status") == "score_only_complete"]
    final_status = "score_only_complete_candidate_reference" if len(good_rows) == 10 else "score_only_partial_failure"
    final_report = {
        **run_manifest,
        "status": final_status,
        "campaign_id": "allen-cahn-score-only-competitive-2026-09-27",
        "results_completed": len(score_rows),
        "successful_endpoints": len(good_rows),
        "expected_endpoints": 10,
        "campaign_wall_seconds_from_first_code_cell": time.perf_counter() - SESSION_STARTED,
        "results": score_rows,
        "scientific_accepted": False,
        "reference_accepted": False
    }
    write_json(SESSION_DIR / "score_only_results.json", final_report)
    csv_fields = (
        "arm", "seed", "post_training_score_status",
        "relative_weighted_space_time_l2", "max_snapshot_relative_l2",
        "max_absolute_error", "heldout_pde_residual_rms_normalized",
        "initial_max_abs", "boundary_max_abs",
        "source_training_status", "reference_accepted", "scientific_accepted"
    )
    with (SESSION_DIR / "score_only_results.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=csv_fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(score_rows)
    torch.cuda.synchronize()
    print(json.dumps({
        "status": final_status,
        "successful_endpoints": len(good_rows),
        "expected_endpoints": 10,
        "source_campaign_root": str(campaign_root),
        "reference_candidate_sha256": reference_sha256,
        "reference_accepted": False,
        "scientific_accepted": False,
        "training_repeated": False,
        "score_only_results_json": str(SESSION_DIR / "score_only_results.json"),
        "score_only_results_csv": str(SESSION_DIR / "score_only_results.csv"),
        "campaign_wall_seconds_from_first_code_cell": time.perf_counter() - SESSION_STARTED,
        "gpu_names": gpu_names
    }, indent=2))
    if len(good_rows) != 10:
        raise RuntimeError("Score-only recovery ended with failed endpoints; partial artifacts were preserved.")

finally:
    watchdog.cancel()